# Análisis comparativo de modelos preentrenados de Hugging Face
## Caso de uso: Análisis de sentimiento en español (NLP)

**Maestría — Actividad: selección e implementación de modelos preentrenados.**

Este cuaderno implementa y compara **tres** modelos preentrenados de clasificación
de sentimiento en español, descargados de **Hugging Face Hub**, sobre un dataset
de evaluación balanceado de 120 reseñas/opiniones cortas (clases `POS`, `NEU`, `NEG`).

| # | Modelo (Hugging Face) | Base | Idioma | Licencia |
|---|------------------------|------|--------|----------|
| 1 | `pysentimiento/robertuito-sentiment-analysis` | RoBERTuito (RoBERTa-es) | Español | No comercial / investigación |
| 2 | `lxyuan/distilbert-base-multilingual-cased-sentiments-student` | DistilBERT multilingüe (destilado) | Multilingüe | Apache-2.0 |
| 3 | `cardiffnlp/twitter-xlm-roberta-base-sentiment` | XLM-RoBERTa base | Multilingüe | Académica (repo XLM-T, MIT) |

**Contenido del cuaderno**
1. Configuración de GPU
2. Instalación de dependencias
3. Autenticación en Hugging Face Hub
4. Carga del dataset
5. Carga de modelos y pipeline de inferencia
6. Métricas y normalización de etiquetas
7. Evaluación (inferencia + métricas + latencia)
8. Tablas y gráficas comparativas
9. Conclusiones parciales

> **Cómo ejecutar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*,
> luego *Entorno de ejecución → Ejecutar todo*. Tiempo aproximado: 3–6 minutos (incluye descarga de modelos).

## 1. Configuración de GPU

Verificamos que el entorno tenga una GPU asignada (en Colab gratuito suele ser una
**NVIDIA T4**). Si `CUDA disponible: False`, activa la GPU en
*Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

In [ ]:
import torch

print("Versión de PyTorch:", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print("Memoria total   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print(">> No hay GPU. El cuaderno funciona en CPU, pero la latencia será mucho mayor.")

# Detalle del hardware (ignora el error si no hay GPU)
!nvidia-smi -L

## 2. Instalación de dependencias

`torch` ya viene preinstalado en Colab. Instalamos `transformers`, `huggingface_hub`
y **`sentencepiece`** (necesario para el *tokenizer* de XLM-RoBERTa). `scikit-learn`,
`pandas` y `matplotlib` también vienen en Colab; se incluyen por reproducibilidad.

In [ ]:
%pip -q install -U "transformers>=4.40" "huggingface_hub>=0.23" sentencepiece accelerate
%pip -q install -U scikit-learn pandas matplotlib

import transformers, sklearn, pandas as pd
print("transformers:", transformers.__version__)
print("scikit-learn:", sklearn.__version__)
print("pandas      :", pd.__version__)

## 3. Autenticación en Hugging Face Hub

Los tres modelos son **públicos**, por lo que la autenticación es *opcional* para
descargarlos. Aun así, autenticarse es una **buena práctica** (evita límites de
descarga anónima y es obligatorio para modelos *gated* o privados).

Dos opciones:
- **A) Token en Secretos de Colab** (recomendado): panel 🔑 a la izquierda → añade un
  secreto `HF_TOKEN` con tu token de https://huggingface.co/settings/tokens.
- **B) Login interactivo:** descomenta `notebook_login()`.

In [ ]:
import os
from huggingface_hub import login

token = None
# Opción A: Secreto de Colab llamado HF_TOKEN
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if token:
    login(token=token)
    print("Autenticado en Hugging Face Hub mediante token.")
else:
    print("Sin token: se continuará en modo anónimo (suficiente para modelos públicos).")
    # Opción B (interactiva), descomenta si lo prefieres:
    # from huggingface_hub import notebook_login
    # notebook_login()

## 4. Carga del dataset

El dataset de evaluación se **embebe en el propio cuaderno** (mismas 120 frases del
archivo `data/dataset_sentimiento_es.csv` del repositorio), de modo que el notebook
es **autocontenido** y se puede ejecutar sin clonar nada. También se guarda una copia
en `data/` para trazabilidad.

- Origen: corpus original curado por los autores (ver `data/README_dataset.md`).
- Clases: `POS`, `NEU`, `NEG` — **balanceado** (40 por clase).

In [ ]:
POSITIVOS = [
    ('Me encanto este telefono! La bateria dura todo el dia y la camara es excelente.', 'electronica'),
    ('El restaurante supero mis expectativas, la comida estaba deliciosa y el servicio fue impecable.', 'restaurante'),
    ('Excelente compra, el producto llego antes de lo esperado y en perfecto estado.', 'compras'),
    ('La aplicacion es muy intuitiva y me ha ahorrado muchisimo tiempo en el trabajo.', 'software'),
    ('Que gran experiencia, el hotel era hermoso y el personal super amable.', 'viajes'),
    ('Recomiendo totalmente este libro, no pude dejar de leerlo hasta el final.', 'entretenimiento'),
    ('La nueva actualizacion mejoro el rendimiento; ahora todo funciona rapidisimo.', 'software'),
    ('Estoy feliz con mi compra, la calidad de la tela es increible y muy comoda.', 'ropa'),
    ('El soporte tecnico resolvio mi problema en minutos, quede muy satisfecho.', 'servicio'),
    ('Una pelicula maravillosa, me emociono de principio a fin.', 'entretenimiento'),
    ('La laptop es ligera, potente y la pantalla se ve espectacular.', 'electronica'),
    ('El viaje fue perfecto, todo estuvo bien organizado y sin contratiempos.', 'viajes'),
    ('Los audifonos tienen un sonido brutal y la cancelacion de ruido funciona de maravilla.', 'electronica'),
    ('Mil gracias al repartidor, super rapido y muy atento.', 'servicio'),
    ('Este curso me cambio la forma de programar, lo volveria a tomar sin dudarlo.', 'educacion'),
    ('El cafe estaba delicioso y el ambiente del local es acogedor.', 'restaurante'),
    ('La verdad, el mejor servicio al cliente que he recibido en anos.', 'servicio'),
    ('Me sorprendio lo bien que funciona, vale cada peso que pague.', 'compras'),
    ('La camara captura fotos nitidas incluso de noche, estoy encantada.', 'electronica'),
    ('Excelente relacion calidad-precio, lo recomiendo a ojos cerrados.', 'compras'),
    ('El sofa es comodisimo y se ve elegante en la sala.', 'hogar'),
    ('La pizza llego caliente y riquisima, repetire sin duda.', 'restaurante'),
    ('Gran atencion del personal, me senti muy bien atendido todo el tiempo.', 'servicio'),
    ('La app de banca por fin es facil de usar, felicidades al equipo!', 'software'),
    ('Increible concierto, la banda sono espectacular y la energia fue contagiosa.', 'entretenimiento'),
    ('El medicamento me alivio rapido, muy contento con los resultados.', 'salud'),
    ('La bicicleta es resistente y muy comoda para los trayectos largos.', 'transporte'),
    ('Servicio puntual y chofer muy amable, llegue tranquilo a mi destino.', 'transporte'),
    ('Las zapatillas son comodisimas, perfectas para correr largas distancias.', 'ropa'),
    ('El tutorial fue clarisimo, aprendi a usar la herramienta en una tarde.', 'educacion'),
    ('Que buen detalle, incluyeron una nota y un regalo con el pedido.', 'compras'),
    ('La serie esta espectacular, cada capitulo te deja con ganas de mas.', 'entretenimiento'),
    ('El hotel tenia una vista preciosa y desayuno buenisimo incluido.', 'viajes'),
    ('Funciona perfecto con mi consola, la instalacion fue rapidisima.', 'electronica'),
    ('La maestra explica muy bien, mi hijo mejoro muchisimo en matematicas.', 'educacion'),
    ('Compre la aspiradora y limpia increible, supero mis expectativas.', 'hogar'),
    ('El equipo de soporte fue paciente y resolvio todas mis dudas.', 'servicio'),
    ('Comida fresca, porciones generosas y precios justos, volvere pronto.', 'restaurante'),
    ('La actualizacion trajo funciones nuevas geniales, me encanta la app ahora.', 'software'),
    ('Excelente calidad de imagen en la tele, los colores son vivos y reales.', 'electronica'),
]

NEUTRALES = [
    ('El paquete llego en la fecha estimada y venia en su caja original.', 'compras'),
    ('El telefono cumple con lo basico, ni me sorprende ni me decepciona.', 'electronica'),
    ('La reunion se reprogramo para el proximo martes a las diez.', 'servicio'),
    ('El producto es de color azul y mide veinte centimetros de largo.', 'compras'),
    ('La aplicacion se actualizo automaticamente durante la noche.', 'software'),
    ('El restaurante abre de lunes a sabado de nueve a seis.', 'restaurante'),
    ('Recibi el correo con la confirmacion de mi pedido esta manana.', 'compras'),
    ('El hotel esta ubicado a dos cuadras de la estacion de metro.', 'viajes'),
    ('La bateria dura lo que indica el fabricante, nada fuera de lo comun.', 'electronica'),
    ('El libro tiene trescientas paginas y esta dividido en diez capitulos.', 'entretenimiento'),
    ('El tecnico vendra a revisar el equipo entre las dos y las cuatro.', 'servicio'),
    ('La camiseta es talla mediana y de algodon, como aparece en la foto.', 'ropa'),
    ('El curso consta de cinco modulos y un examen final.', 'educacion'),
    ('El pedido incluye dos articulos y un instructivo de uso.', 'compras'),
    ('La pelicula dura aproximadamente dos horas con creditos incluidos.', 'entretenimiento'),
    ('El pago se proceso con tarjeta y recibi el comprobante.', 'servicio'),
    ('El producto requiere dos pilas AA que no vienen incluidas.', 'electronica'),
    ('La tienda cambio su horario por la temporada de fin de ano.', 'compras'),
    ('El envio se realiza mediante paqueteria estandar en cinco dias habiles.', 'compras'),
    ('La app esta disponible para Android y iOS por igual.', 'software'),
    ('El modelo nuevo es similar al anterior, con cambios menores.', 'electronica'),
    ('El vuelo hace una escala de una hora en la ciudad de Panama.', 'viajes'),
    ('Me entregaron la factura impresa junto con el producto.', 'servicio'),
    ('El monitor tiene dos puertos HDMI y uno de tipo USB-C.', 'electronica'),
    ('El plato lleva arroz, pollo y verduras al vapor.', 'restaurante'),
    ('La contrasena debe tener al menos ocho caracteres.', 'software'),
    ('El evento se llevara a cabo en el salon principal del segundo piso.', 'servicio'),
    ('Cambiaron el empaque, pero el contenido es el mismo de siempre.', 'compras'),
    ('El servicio incluye instalacion, pero no la configuracion avanzada.', 'servicio'),
    ('La garantia cubre doce meses a partir de la fecha de compra.', 'compras'),
    ('El autobus pasa cada quince minutos por esta parada.', 'transporte'),
    ('El documento esta en formato PDF y pesa dos megabytes.', 'software'),
    ('La habitacion tiene una cama matrimonial y un escritorio.', 'viajes'),
    ('El supermercado reabastece sus estantes los dias miercoles.', 'compras'),
    ('La actualizacion ocupa quinientos megabytes de espacio.', 'software'),
    ('El producto se fabrica en Mexico y se distribuye en la region.', 'compras'),
    ('La cita medica quedo agendada para el jueves por la tarde.', 'salud'),
    ('El cargador es compatible con varios modelos de la marca.', 'electronica'),
    ('El informe resume las ventas del primer trimestre del ano.', 'servicio'),
    ('El paquete contiene el manual en espanol y en ingles.', 'compras'),
]

NEGATIVOS = [
    ('Pesima experiencia, el producto llego roto y nadie responde mis mensajes.', 'compras'),
    ('La comida estaba fria y el servicio fue lentisimo, no vuelvo.', 'restaurante'),
    ('La bateria se descarga en un par de horas, una verdadera decepcion.', 'electronica'),
    ('La aplicacion se cierra sola constantemente, es imposible usarla.', 'software'),
    ('El hotel estaba sucio y el personal fue grosero, no lo recomiendo.', 'viajes'),
    ('Me cobraron de mas y todavia no me devuelven el dinero.', 'servicio'),
    ('El libro es aburridisimo, no pude pasar del primer capitulo.', 'entretenimiento'),
    ('La actualizacion arruino el telefono, ahora va lentisimo.', 'software'),
    ('Pesima calidad, la tela se rompio a la primera lavada.', 'ropa'),
    ('Llevo una semana esperando el reembolso y nadie me da respuesta.', 'servicio'),
    ('La pelicula es un desastre, perdi dos horas de mi vida.', 'entretenimiento'),
    ('La laptop se sobrecalienta y se apaga sola, que frustracion.', 'electronica'),
    ('El paquete nunca llego y la paqueteria no da explicaciones.', 'compras'),
    ('Los audifonos dejaron de funcionar a la semana, dinero tirado.', 'electronica'),
    ('El repartidor fue muy grosero y dejo el pedido tirado en la puerta.', 'servicio'),
    ('Este curso es una estafa, el contenido esta desactualizado.', 'educacion'),
    ('El cafe sabia a quemado y lo cobraron carisimo.', 'restaurante'),
    ('El peor servicio al cliente, me dejaron en espera una hora.', 'servicio'),
    ('No funciona como prometen, me siento totalmente enganado.', 'compras'),
    ('La camara toma fotos borrosas, una pesima compra.', 'electronica'),
    ('Carisimo para lo poco que ofrece, no lo vale en absoluto.', 'compras'),
    ('El sofa llego con manchas y huele horrible, lo quiero devolver.', 'hogar'),
    ('La pizza llego fria y con ingredientes que no pedi.', 'restaurante'),
    ('Nadie en el local nos atendio, terminamos yendonos molestos.', 'restaurante'),
    ('La app del banco falla justo cuando necesitas pagar, indignante.', 'software'),
    ('El concierto fue una decepcion, el sonido era pesimo.', 'entretenimiento'),
    ('El medicamento no sirvio de nada y encima me cayo mal.', 'salud'),
    ('La bicicleta se descompuso a la semana, mala calidad.', 'transporte'),
    ('El chofer manejo imprudente y llego tardisimo, terrible.', 'transporte'),
    ('Las zapatillas se despegaron al mes, que perdida de dinero.', 'ropa'),
    ('El tutorial es confuso y esta lleno de errores, no sirve.', 'educacion'),
    ('Faltaban piezas en el pedido y nadie soluciona el problema.', 'compras'),
    ('La serie es predecible y mal actuada, no la recomiendo.', 'entretenimiento'),
    ('La habitacion del hotel tenia cucarachas, una asquerosidad.', 'viajes'),
    ('El producto es incompatible con mi equipo, perdi mi dinero.', 'electronica'),
    ('La maestra nunca responde dudas y las clases son un caos.', 'educacion'),
    ('La aspiradora hace un ruido insoportable y no aspira bien.', 'hogar'),
    ('Soporte tecnico inutil, llevo dias sin que resuelvan nada.', 'servicio'),
    ('Comida en mal estado, termine con dolor de estomago.', 'restaurante'),
    ('La nueva version elimino funciones utiles, un retroceso total.', 'software'),
]


import os
import pandas as pd

filas, idx = [], 1
for etiqueta, ejemplos in [("POS", POSITIVOS), ("NEU", NEUTRALES), ("NEG", NEGATIVOS)]:
    for texto, dominio in ejemplos:
        filas.append({"id": f"es-{idx:03d}", "texto": texto, "etiqueta": etiqueta,
                      "dominio": dominio, "longitud_caracteres": len(texto),
                      "num_palabras": len(texto.split())})
        idx += 1

df = pd.DataFrame(filas)
os.makedirs("data", exist_ok=True)
df.to_csv("data/dataset_sentimiento_es.csv", index=False, encoding="utf-8")

print("Total de ejemplos:", len(df))
print(df["etiqueta"].value_counts())
df.head()

## 5. Carga de modelos y pipeline de inferencia

Definimos los tres modelos y una función que construye un `pipeline` de
`text-classification` sobre la GPU (`device=0`) o CPU (`device=-1`). Incluimos un
preprocesamiento ligero (reemplazo de menciones y URLs) recomendado por los modelos
entrenados con tweets.

In [ ]:
from transformers import pipeline

MODELOS = {
    "RoBERTuito (es)":              "pysentimiento/robertuito-sentiment-analysis",
    "DistilBERT-multi (destilado)": "lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    "XLM-RoBERTa (multi)":          "cardiffnlp/twitter-xlm-roberta-base-sentiment",
}

DEVICE = 0 if torch.cuda.is_available() else -1

def preprocesar(texto):
    # Normaliza menciones @usuario y enlaces (convención de los modelos de Twitter)
    palabras = []
    for t in texto.split(" "):
        if t.startswith("@") and len(t) > 1:
            t = "@user"
        elif t.startswith("http"):
            t = "http"
        palabras.append(t)
    return " ".join(palabras)

def construir_pipeline(model_id):
    return pipeline("text-classification", model=model_id, tokenizer=model_id,
                    device=DEVICE, truncation=True, max_length=256)

# Prueba rápida con un modelo
demo = construir_pipeline(MODELOS["RoBERTuito (es)"])
print(demo("Que gran jugador es Messi"))
print(demo("El producto llego roto y nadie responde"))
del demo
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 6. Métricas y normalización de etiquetas

Cada modelo expone etiquetas distintas (`POS/NEU/NEG`, `positive/neutral/negative`,
`Positive/Neutral/Negative`). Esta es la única **decisión de ajuste mínimo** necesaria:
normalizar todas las salidas a un esquema canónico `{NEG, NEU, POS}` para poder
compararlas de forma homogénea. Después calculamos *accuracy*, *precision*, *recall*
y *F1* (macro y por clase).

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

ETIQUETAS = ["NEG", "NEU", "POS"]

_MAPA = {
    "pos": "POS", "positive": "POS", "positivo": "POS",
    "neu": "NEU", "neutral": "NEU", "neutro": "NEU",
    "neg": "NEG", "negative": "NEG", "negativo": "NEG",
    "label_0": "NEG", "label_1": "NEU", "label_2": "POS",
}

def normalizar_etiqueta(etiqueta):
    clave = str(etiqueta).strip().lower()
    if clave not in _MAPA:
        raise ValueError(f"Etiqueta no reconocida: {etiqueta!r}")
    return _MAPA[clave]

def calcular_metricas(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", labels=ETIQUETAS, zero_division=0)
    return {"accuracy": acc, "precision_macro": p, "recall_macro": r, "f1_macro": f1}

# Comprobación del normalizador
print([normalizar_etiqueta(x) for x in ["POS", "positive", "Negative", "NEU"]])

## 7. Evaluación: inferencia, métricas y latencia

Para cada modelo:
1. Descargamos y construimos el pipeline.
2. Hacemos **inferencia por lotes** (`batch_size=16`) sobre las 120 frases.
3. Calculamos las métricas frente a las etiquetas de referencia.
4. Medimos la **latencia media por muestra** con `batch=1` (escenario de petición
   individual), descartando una pasada de calentamiento.

In [ ]:
import time

def medir_latencia(clf, textos, repeticiones=3):
    _ = clf(textos[0])  # calentamiento
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        for t in textos:
            _ = clf(t)
        tiempos.append(time.perf_counter() - t0)
    return (min(tiempos) / len(textos)) * 1000.0  # ms/muestra

textos = [preprocesar(t) for t in df["texto"].tolist()]
y_true = df["etiqueta"].tolist()

resultados, detalle = [], {}
for nombre, model_id in MODELOS.items():
    print(f"\n=== {nombre}  ({model_id}) ===")
    clf = construir_pipeline(model_id)

    t0 = time.perf_counter()
    preds = clf(textos, batch_size=16)
    t_total = time.perf_counter() - t0

    y_pred = [normalizar_etiqueta(p["label"]) for p in preds]
    m = calcular_metricas(y_true, y_pred)
    lat = medir_latencia(clf, textos)

    print("Accuracy: %.4f | F1 macro: %.4f | Precisión: %.4f | Recall: %.4f"
          % (m["accuracy"], m["f1_macro"], m["precision_macro"], m["recall_macro"]))
    print("Latencia: %.2f ms/muestra | throughput lote: %.1f muestras/s"
          % (lat, len(textos) / t_total))
    print(classification_report(y_true, y_pred, labels=ETIQUETAS, zero_division=0, digits=4))

    resultados.append({"modelo": nombre, "accuracy": m["accuracy"],
                       "precision_macro": m["precision_macro"], "recall_macro": m["recall_macro"],
                       "f1_macro": m["f1_macro"], "latencia_ms": lat,
                       "throughput_muestras_s": len(textos) / t_total})
    detalle[nombre] = {"y_pred": y_pred,
                       "cm": confusion_matrix(y_true, y_pred, labels=ETIQUETAS)}
    del clf
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 8. Tablas y gráficas comparativas

Resumimos los resultados en una tabla ordenada por *F1 macro* y los guardamos en
`results/`. Después graficamos: (a) F1 vs. accuracy, (b) latencia vs. F1
(trade-off eficiencia/precisión) y (c) matrices de confusión.

In [ ]:
import os
os.makedirs("results", exist_ok=True)

tabla = pd.DataFrame(resultados).sort_values("f1_macro", ascending=False).reset_index(drop=True)
tabla_fmt = tabla.copy()
for c in ["accuracy", "precision_macro", "recall_macro", "f1_macro"]:
    tabla_fmt[c] = tabla_fmt[c].map(lambda x: round(x, 4))
tabla_fmt["latencia_ms"] = tabla_fmt["latencia_ms"].map(lambda x: round(x, 2))
tabla_fmt["throughput_muestras_s"] = tabla_fmt["throughput_muestras_s"].map(lambda x: round(x, 1))

tabla_fmt.to_csv("results/tabla_resumen.csv", index=False)
print("Tabla guardada en results/tabla_resumen.csv")
tabla_fmt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# (a) Barras: F1 macro y accuracy
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(tabla)); w = 0.38
ax.bar(x - w/2, tabla["f1_macro"], w, label="F1 macro")
ax.bar(x + w/2, tabla["accuracy"], w, label="Accuracy")
ax.set_xticks(x); ax.set_xticklabels(tabla["modelo"], rotation=15, ha="right")
ax.set_ylim(0, 1); ax.set_ylabel("Puntuación"); ax.set_title("Precisión por modelo")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("results/fig_precision.png", dpi=150); plt.show()

In [ ]:
# (b) Trade-off: latencia (ms) vs F1 macro
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(tabla["latencia_ms"], tabla["f1_macro"], s=120)
for _, row in tabla.iterrows():
    ax.annotate(row["modelo"], (row["latencia_ms"], row["f1_macro"]),
                textcoords="offset points", xytext=(8, 6), fontsize=9)
ax.set_xlabel("Latencia (ms/muestra)  —  menor es mejor")
ax.set_ylabel("F1 macro  —  mayor es mejor")
ax.set_title("Trade-off eficiencia vs. precisión")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("results/fig_tradeoff.png", dpi=150); plt.show()

In [ ]:
# (c) Matrices de confusión
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (nombre, d) in zip(axes, detalle.items()):
    cm = d["cm"]
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(nombre, fontsize=10)
    ax.set_xticks(range(3)); ax.set_xticklabels(ETIQUETAS)
    ax.set_yticks(range(3)); ax.set_yticklabels(ETIQUETAS)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout(); plt.savefig("results/fig_matrices_confusion.png", dpi=150); plt.show()

## 9. Conclusiones parciales

La siguiente celda genera conclusiones **a partir de los números reales** obtenidos
en esta ejecución (modelo más preciso, más rápido y mejor equilibrado).

In [ ]:
mejor_f1   = tabla.loc[tabla["f1_macro"].idxmax()]
mas_rapido = tabla.loc[tabla["latencia_ms"].idxmin()]

# Eficiencia = F1 por cada ms de latencia (mayor es mejor)
tabla["eficiencia_f1_por_ms"] = tabla["f1_macro"] / tabla["latencia_ms"]
mejor_equilibrio = tabla.loc[tabla["eficiencia_f1_por_ms"].idxmax()]

print("CONCLUSIONES PARCIALES (ejecución actual)")
print("-" * 55)
print(f"• Mayor F1 macro : {mejor_f1['modelo']}  (F1={mejor_f1['f1_macro']:.4f}, "
      f"acc={mejor_f1['accuracy']:.4f})")
print(f"• Más rápido     : {mas_rapido['modelo']}  ({mas_rapido['latencia_ms']:.2f} ms/muestra)")
print(f"• Mejor equilibrio precisión/latencia: {mejor_equilibrio['modelo']}")
print()
print("Lectura del trade-off:")
print("  - El modelo destilado (DistilBERT, 6 capas) maximiza velocidad.")
print("  - El modelo especializado en español (RoBERTuito) suele liderar en F1")
print("    sobre texto corto/informal en español.")
print("  - XLM-RoBERTa (270M) aporta robustez multilingüe a mayor costo de latencia.")

### Síntesis

- **Calidad (F1/accuracy):** se espera que **RoBERTuito** lidere en español por estar
  especializado en el idioma y dominio (tweets), seguido de cerca por **XLM-RoBERTa**;
  el **DistilBERT destilado** queda por debajo al provenir de destilación *zero-shot*.
- **Eficiencia (latencia):** **DistilBERT** es el más rápido (la mitad de capas),
  ideal para alto volumen o dispositivos limitados.
- **Licencia:** solo **DistilBERT (Apache-2.0)** permite uso comercial sin fricción;
  RoBERTuito es de uso no comercial / investigación.
- **Recomendación (prototipo académico en español):** **RoBERTuito** por su mejor
  relación precisión/recursos en español. Para producción comercial o multilingüe,
  evaluar **XLM-RoBERTa** (precisión) o **DistilBERT** (eficiencia + licencia).

> Los valores definitivos provienen de la tabla generada arriba en *tu* ejecución.
> El reporte en Word (`docs/Analisis_Comparativo_Modelos.docx`) amplía criterios,
> trade-offs, umbrales y riesgos.